### Scott 10K demographics

In [18]:
import pandas as pd
import scipy
from scipy import stats
import numpy as np
import pandas as pd
from scipy.stats import chisquare, f_oneway

df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', low_memory =False)

# Ensure we only have valid rows with VBM data
df = df.dropna(subset = [col for col in df.columns.tolist() if 'desikan_vbm' in col])
df = df.dropna(subset = 'DIAGNOSIS')

print(df.shape)

(11926, 928)


In [102]:
def chi_sq(col: str, flag : str, subsets = True, df = df.copy()):
    observed_counts = df[col].value_counts().sort_index()
    expected_counts = [observed_counts.sum() / len(observed_counts)] * len(observed_counts)
    chi2_stat, p_val = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    print(flag)
    print(observed_counts)
    print(f"Total number in {col} = {len(df[col].dropna())}")
    print(f"Chi² = {chi2_stat:.4f}, p = {p_val:.4f}\n")
    if subsets:
        for diag in [1,2,3]:
            df_sub = df[df['DIAGNOSIS'] == diag]
            scanner_counts = df_sub[col].value_counts().sort_index()
            expected_counts = [scanner_counts.sum() / len(scanner_counts)] * len(scanner_counts)
            print(diag)
            print(df_sub[col].value_counts().sort_index())
            chi2_stat, p_val = chisquare(f_obs=scanner_counts, f_exp=expected_counts)
            print(f"Chi² {diag} = {chi2_stat:.4f}, p = {p_val:.4f}\n")
            print('')

In [103]:
chi_sq(col = 'DIAGNOSIS', flag ='=== Diagnosis Distribution ===', subsets = False)

=== Diagnosis Distribution ===
DIAGNOSIS
1.0    4035
2.0    5056
3.0    2835
Name: count, dtype: int64
Total number in DIAGNOSIS = 11926
Chi² = 621.7744, p = 0.0000



In [104]:
for col, flag in [('FIELD_STRENGTH', '=== Scanner Field Strength ==='), ('PTGENDER', "=== Gender Distribution ===")]:
    chi_sq(col = col, flag = flag)

=== Scanner Field Strength ===
FIELD_STRENGTH
1.5T    5361
3T      6556
Name: count, dtype: int64
Total number in FIELD_STRENGTH = 11917
Chi² = 119.8309, p = 0.0000

1
FIELD_STRENGTH
1.5T    1657
3T      2375
Name: count, dtype: int64
Chi² 1 = 127.8581, p = 0.0000


2
FIELD_STRENGTH
1.5T    2077
3T      2979
Name: count, dtype: int64
Chi² 2 = 160.9185, p = 0.0000


3
FIELD_STRENGTH
1.5T    1627
3T      1202
Name: count, dtype: int64
Chi² 3 = 63.8476, p = 0.0000


=== Gender Distribution ===
PTGENDER
1.0    6351
2.0    5266
Name: count, dtype: int64
Total number in PTGENDER = 11617
Chi² = 101.3364, p = 0.0000

1
PTGENDER
1.0    1824
2.0    2130
Name: count, dtype: int64
Chi² 1 = 23.6813, p = 0.0000


2
PTGENDER
1.0    2970
2.0    1971
Name: count, dtype: int64
Chi² 2 = 201.9836, p = 0.0000


3
PTGENDER
1.0    1557
2.0    1165
Name: count, dtype: int64
Chi² 3 = 56.4526, p = 0.0000




In [128]:
from scipy.stats import chisquare

def chi_sq_ptid(flag: str, df=df.copy(), col='PTID'):
    observed_counts = df.groupby('DIAGNOSIS')[col].nunique()
    expected_counts = [observed_counts.sum() / len(observed_counts)] * len(observed_counts)
    chi2_stat, p_val = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    print(flag)
    print(f"Total number in {col} = {len(df[col].dropna())}")
    print(f"Chi² = {chi2_stat:.4f}, p = {p_val:.4f}\n")
    for diag in [1,2,3]:
        df_sub = df[df['DIAGNOSIS'] == diag]
        print(f'Number of patient IDs for diagnosis = {diag}: {df_sub['PTID'].nunique()}')

chi_sq_ptid('PTID')

PTID
Total number in PTID = 11926
Chi² = 117.7212, p = 0.0000

Number of patient IDs for diagnosis = 1: 983
Number of patient IDs for diagnosis = 2: 1223
Number of patient IDs for diagnosis = 3: 742


In [42]:
import pickle

def chi_sq_sbm(file: str, flag : str, df = df.copy()):

    loaded_cols = pickle.load(open(f'/rds/general/project/c3nl_scott_students/live/sankeith/standards/{file}.pkl', 'rb'))
    print(f'Preparing Chi² analysis for {flag}')
    df_sub = df[['DIAGNOSIS'] + loaded_cols].dropna(subset = loaded_cols)
    print(df_sub.shape)
    col = loaded_cols[0]
    observed_counts = df[col].value_counts().sort_index() # Using the first element in loaded_cols works: with recon-all, it's either all values are there or none are there. We've used dropna anyway so it should be good
    print(observed_counts)
    expected_counts = [observed_counts.sum() / len(observed_counts)] * len(observed_counts)
    chi2_stat, p_val = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    print(f"Total number of recorded diagnoses = {len(df[col].dropna())}")
    print(f"Chi² = {chi2_stat:.4f}, p = {p_val:.4f}\n")

In [43]:
for file, flag in [('ordered_desikan_sbm_cort_thick_cols', 'Desikan SBM cortical thickness'), ('ordered_desikan_sbm_roi_gmv_cols', 'Desikan ROI GMV')]:
    chi_sq_sbm(file, flag)

Preparing Chi² analysis for Desikan SBM cortical thickness
(11890, 70)
DIAGNOSIS
1.0    4035
2.0    5056
3.0    2835
Name: count, dtype: int64
Total number of recorded diagnoses = 11926
Chi² = 621.7744, p = 0.0000

Preparing Chi² analysis for Desikan ROI GMV
(11890, 88)
DIAGNOSIS
1.0    4035
2.0    5056
3.0    2835
Name: count, dtype: int64
Total number of recorded diagnoses = 11926
Chi² = 621.7744, p = 0.0000



In [70]:
def myonewayanova(col, flag: str, df = df.copy()):
    vals_by_diag = {}
    print(flag)
    print('')
    for diag in [1,2,3]:
        df_sub = df[df['DIAGNOSIS'] == diag]
        print(f'Diagnosis = {diag} \n{df_sub[col].describe()}')
        print('')
        vals = pd.to_numeric(df_sub[col], errors='coerce').dropna()
        print(vals.shape)
        vals_by_diag[diag] = vals
    anova = f_oneway(*vals_by_diag.values())
    print(f"One-Way ANOVA on {flag}: F({len(vals_by_diag)-1}, {sum(len(v) for v in vals_by_diag.values()) - len(vals_by_diag)}) = {anova.statistic:.4f}, p = {anova.pvalue:.4f}")
    print('')
    

In [71]:
for col, flag in [('PTAGE', '=== Age distribution ==='), 
                  ('NPISCORE', '=== NPI-Q score distribution ==='), 
                  ('CDGLOBAL', '=== CDR score distribution ==='), 
                  ('MMSCORE', '=== MMSE score distribution'),
                  ('Abeta_40_conc', '=== Amyloid Beta 40 concentration distribution ==='),
                  ('Abeta_42_conc', '=== Amyloid Beta 42 concentration distribution ==='),
                  ('P217_DILUTION_CORRECTED_CONC', '=== p-217 tau concentration distribution ===')]:
    myonewayanova(col = col, flag = flag)
                                    

=== Age distribution ===

Diagnosis = 1 
count    3954.000000
mean       75.147193
std         6.632035
min        53.000000
25%        71.000000
50%        75.000000
75%        79.000000
max       102.000000
Name: PTAGE, dtype: float64

(3954,)
Diagnosis = 2 
count    4941.000000
mean       74.486744
std         7.635555
min        55.000000
25%        69.000000
50%        75.000000
75%        80.000000
max        93.000000
Name: PTAGE, dtype: float64

(4941,)
Diagnosis = 3 
count    2722.000000
mean       75.893461
std         7.533219
min        55.000000
25%        71.000000
50%        76.000000
75%        81.000000
max        94.000000
Name: PTAGE, dtype: float64

(2722,)
One-Way ANOVA on === Age distribution ===: F(2, 11614) = 33.3603, p = 0.0000

=== NPI-Q score distribution ===

Diagnosis = 1 
count    1942.000000
mean        0.587024
std         1.395288
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        11.000000
Name: NPISCORE, dty

#### Crystal ball dataset demographics

In [96]:
import pandas as pd
import numpy as np
from datetime import datetime
cb_df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/cb_df_far.csv', low_memory =False)
print(cb_df.columns.tolist())
cb_df['VISDATE_diff'] = pd.to_timedelta(cb_df['VISDATE_diff'])
cb_df['VISDATE_diff'] = cb_df['VISDATE_diff'].dt.days
print(f'DIAGNOSIS COUNTS (converters vs non-converters: \n{cb_df['DIAGNOSIS_oth'].value_counts()}')

['PTID', 'DATA_KEY', 'VISDATE_mci', 'PTAGE_mci', 'PTGENDER', 'FIELD_STRENGTH_mci', 'DIAGNOSIS_mci', 'MMSCORE_mci', 'CDGLOBAL_mci', 'NPISCORE_mci', 'desikan_D1_mci', 'desikan_D2_mci', 'desikan_DAT_mci', 'desikan_NET_mci', 'desikan_5HT1A_mci', 'desikan_5HT1B_mci', 'desikan_5HT2A_mci', 'desikan_5HT4_mci', 'desikan_5HT6_mci', 'desikan_5HTT_mci', 'desikan_a4b2_mci', 'desikan_M1_mci', 'desikan_vAChT_mci', 'desikan_NMDA_mci', 'desikan_mGluR5_mci', 'desikan_GABAA/BZ_mci', 'desikan_H3_mci', 'desikan_CB1_mci', 'desikan_MOR_mci', 'VISDATE_oth', 'DIAGNOSIS_oth', 'MMSCORE_oth', 'CDGLOBAL_oth', 'NPISCORE_oth', 'desikan_D1_oth', 'desikan_D2_oth', 'desikan_DAT_oth', 'desikan_NET_oth', 'desikan_5HT1A_oth', 'desikan_5HT1B_oth', 'desikan_5HT2A_oth', 'desikan_5HT4_oth', 'desikan_5HT6_oth', 'desikan_5HTT_oth', 'desikan_a4b2_oth', 'desikan_M1_oth', 'desikan_vAChT_oth', 'desikan_NMDA_oth', 'desikan_mGluR5_oth', 'desikan_GABAA/BZ_oth', 'desikan_H3_oth', 'desikan_CB1_oth', 'desikan_MOR_oth', 'VISDATE_diff', 'M

In [129]:
def chi_sq(col: str, flag : str, subsets = True, df = cb_df.copy()):
    observed_counts = df[col].value_counts().sort_index()
    expected_counts = [observed_counts.sum() / len(observed_counts)] * len(observed_counts)
    chi2_stat, p_val = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    print(flag)
    print(observed_counts)
    print(f"Total number of recorded diagnoses = {len(df[col].dropna())}")
    print(f"Chi² = {chi2_stat:.4f}, p = {p_val:.4f}\n")
    if subsets:
        for diag in [0,1]:
            df_sub = df[df['DIAGNOSIS_oth'] == diag]
            scanner_counts = df_sub[col].value_counts().sort_index()
            expected_counts = [scanner_counts.sum() / len(scanner_counts)] * len(scanner_counts)
            print(diag)
            print(df_sub[col].value_counts().sort_index())
            chi2_stat, p_val = chisquare(f_obs=scanner_counts, f_exp=expected_counts)
            print(f"Chi² {diag} = {chi2_stat:.4f}, p = {p_val:.4f}\n")
            print('')

def myonewayanova(col, flag: str, df = cb_df.copy()):
    vals_by_diag = {}
    print(flag)
    print('')
    for diag in [0,1]:
        df_sub = df[df['DIAGNOSIS_oth'] == diag]
        print(f'Diagnosis = {diag} \n{df_sub[col].describe()}')
        print('')
        vals = pd.to_numeric(df_sub[col], errors='coerce').dropna()
        print(vals.shape)
        vals_by_diag[diag] = vals
    anova = f_oneway(*vals_by_diag.values())
    print(f"One-Way ANOVA on {flag}: F({len(vals_by_diag)-1}, {sum(len(v) for v in vals_by_diag.values()) - len(vals_by_diag)}) = {anova.statistic:.4f}, p = {anova.pvalue:.4f}")
    print('')

def chi_sq_ptid(flag: str, df=cb_df.copy(), col='PTID'):
    observed_counts = df.groupby('DIAGNOSIS_oth')[col].nunique()
    expected_counts = [observed_counts.sum() / len(observed_counts)] * len(observed_counts)
    chi2_stat, p_val = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    print(flag)
    print(f"Total number in {col} = {len(df[col].dropna())}")
    print(f"Chi² = {chi2_stat:.4f}, p = {p_val:.4f}\n")
    for diag in [0, 1]:
        df_sub = df[df['DIAGNOSIS_oth'] == diag]
        print(f'Number of patient IDs for diagnosis = {diag}: {df_sub['PTID'].nunique()}')


    

In [130]:

for col, flag in [('PTGENDER', '=== Age distribution (crystal ball) ==='),
                 ('FIELD_STRENGTH_mci', '=== Scanner strength (crystal ball) ===')]:
    chi_sq(col = col, flag = flag)

for col, flag in [('PTAGE_mci', '==== Crystalball age distribution (at time of MCI diagnosis) ===='), 
                  ('VISDATE_diff', '=== Number of days between baseline and most recent visit ==='),
                  ('MMSCORE_diff', '=== Change in MMSE score between baseline and most recent visit ==='),
                 ('NPISCORE_diff', '=== Change in NPI-Q score between baseline and most recent visit ==='),
                 ('CDGLOBAL_diff', '=== Change in CDR score between baseline and most recent visit ===')]:
    myonewayanova(col = col, flag = flag)

chi_sq_ptid('PTID')

=== Age distribution (crystal ball) ===
PTGENDER
1.0    568
2.0    385
Name: count, dtype: int64
Total number of recorded diagnoses = 953
Chi² = 35.1406, p = 0.0000

0
PTGENDER
1.0    384
2.0    264
Name: count, dtype: int64
Chi² 0 = 22.2222, p = 0.0000


1
PTGENDER
1.0    184
2.0    121
Name: count, dtype: int64
Chi² 1 = 13.0131, p = 0.0003


=== Scanner strength (crystal ball) ===
FIELD_STRENGTH_mci
1.5T    359
3T      602
Name: count, dtype: int64
Total number of recorded diagnoses = 961
Chi² = 61.4454, p = 0.0000

0
FIELD_STRENGTH_mci
1.5T    186
3T      466
Name: count, dtype: int64
Chi² 0 = 120.2454, p = 0.0000


1
FIELD_STRENGTH_mci
1.5T    173
3T      136
Name: count, dtype: int64
Chi² 1 = 4.4304, p = 0.0353


==== Crystalball age distribution (at time of MCI diagnosis) ====

Diagnosis = 0 
count    648.000000
mean      73.063272
std        7.695962
min       55.000000
25%       68.000000
50%       73.000000
75%       79.000000
max       91.000000
Name: PTAGE_mci, dtype: float6